In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

df_final = pd.read_csv("../data/processed/cleaned_beja.csv")
df_final['Date'] = pd.to_datetime(df_final['Date'])

print(df_final.shape)
print(df_final.columns.tolist())

(667, 11)
['source_file', 'Date', 'variety', 'Nuages_%', 'NDVI', 'EVI', 'NDRE', 'GNDVI', 'SAVI', 'decision', 'was_originally_missing']


In [2]:
reliable_mask = ~df_final['was_originally_missing']
train_df = df_final[reliable_mask].copy()

interpolated_mask = (df_final['was_originally_missing']) & (df_final['decision'] == 'INTERPOLATE')
predict_df = df_final[interpolated_mask].copy()

print(f"Training rows (trusted): {len(train_df)}")
print(f"Rows to predict/validate (interpolated): {len(predict_df)}")
print(f"Any NaNs in training features?\n{train_df[['EVI','NDRE','GNDVI','SAVI','NDVI']].isna().sum()}")

Training rows (trusted): 627
Rows to predict/validate (interpolated): 16
Any NaNs in training features?
EVI      0
NDRE     0
GNDVI    0
SAVI     0
NDVI     0
dtype: int64


In [3]:
# turn Date into something numeric: day of year captures seasonality
train_df['day_of_year'] = train_df['Date'].dt.dayofyear
predict_df['day_of_year'] = predict_df['Date'].dt.dayofyear

# encode variety (text) into numbers
le_variety = LabelEncoder()
train_df['variety_encoded'] = le_variety.fit_transform(train_df['variety'])
predict_df['variety_encoded'] = le_variety.transform(predict_df['variety'])

feature_cols = ['EVI', 'NDRE', 'GNDVI', 'SAVI', 'Nuages_%', 'day_of_year', 'variety_encoded']

X_train = train_df[feature_cols]
y_train = train_df['NDVI']

X_predict = predict_df[feature_cols]

print(X_train.shape, y_train.shape, X_predict.shape)
print(X_train.head())

(627, 7) (627,) (16, 7)
        EVI      NDRE     GNDVI      SAVI  Nuages_%  day_of_year  \
0  0.075329  0.082779  0.277472  0.082654      1.23          308   
1  0.089692  0.100240  0.317070  0.099258     12.33          313   
2  0.097429  0.109707  0.327559  0.109638      2.22          318   
3  0.126662  0.153942  0.368445  0.139012     10.94          328   
4  0.199998  0.013696  0.031475  0.039936     17.39          333   

   variety_encoded  
0                0  
1                0  
2                0  
3                0  
4                0  


In [4]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Model trained.")
print(f"Training R² score: {rf_model.score(X_train, y_train):.4f}")

Model trained.
Training R² score: 0.9992


In [5]:
rf_predictions = rf_model.predict(X_predict)

comparison_rf = predict_df[['source_file', 'Date', 'NDVI']].copy()
comparison_rf = comparison_rf.rename(columns={'NDVI': 'NDVI_curvefit'})  # this column currently holds your Layer-1 final values
comparison_rf['NDVI_rf_predicted'] = rf_predictions
comparison_rf['diff_rf_vs_curvefit'] = comparison_rf['NDVI_rf_predicted'] - comparison_rf['NDVI_curvefit']

print(comparison_rf.to_string())
print(f"\nMean absolute difference (RF vs curve-fit): {comparison_rf['diff_rf_vs_curvefit'].abs().mean():.4f}")

                       source_file       Date  NDVI_curvefit  NDVI_rf_predicted  diff_rf_vs_curvefit
11      indices_nuages_carioka.csv 2025-04-04       0.774801           0.779880             0.005079
120     indices_nuages_dhahbi6.csv 2024-12-13       0.309166           0.280026            -0.029141
172    indices_nuages_inrat100.csv 2025-04-04       0.568779           0.606734             0.037954
195  indices_nuages_inrat100_1.csv 2025-04-04       0.892366           0.908790             0.016424
218  indices_nuages_inrat100_2.csv 2025-04-04       0.788066           0.811587             0.023521
241       indices_nuages_karim.csv 2025-04-04       0.817031           0.818565             0.001534
264      indices_nuages_karim1.csv 2025-04-04       0.820660           0.837074             0.016414
310      indices_nuages_khiar1.csv 2025-04-04       0.849234           0.854206             0.004972
333      indices_nuages_khiar2.csv 2025-04-04       0.767802           0.792118            

CROSS-VALIDATION


In [6]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='r2')

print(f"Cross-validation R² scores (5 folds): {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f}")
print(f"Std CV R²: {cv_scores.std():.4f}")

Cross-validation R² scores (5 folds): [0.99485013 0.9907044  0.99512121 0.99476951 0.99441889]
Mean CV R²: 0.9940
Std CV R²: 0.0016
